In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications.vgg16 import preprocess_input
import matplotlib.pyplot as plt
import numpy as np

# 1. Load CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# 2. Resize to 224x224 and normalize using VGG preprocessing
x_train = tf.image.resize(x_train, (224, 224))
x_test = tf.image.resize(x_test, (224, 224))
x_train = preprocess_input(x_train)
x_test = preprocess_input(x_test)

# 3. One-hot encode labels
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

# 4. Load VGG16 base model without the top layer
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze base model layers
for layer in base_model.layers:
    layer.trainable = False

# 5. Add custom classification head
x = base_model.output
x = Flatten()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(10, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

# 6. Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# 7. Train the model (use small epochs for demo)
model.fit(x_train, y_train_cat, epochs=5, batch_size=32, validation_data=(x_test, y_test_cat))

# 8. Evaluate
test_loss, test_acc = model.evaluate(x_test, y_test_cat, verbose=2)
print(f"\n✅ Test Accuracy: {test_acc:.4f}")

# 9. Predict and show some results
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

def show_predictions(model, x_test, y_test, class_names, num_images=5):
    predictions = model.predict(x_test[:num_images])
    pred_labels = np.argmax(predictions, axis=1)
    true_labels = y_test[:num_images].flatten()

    plt.figure(figsize=(15, 5))
    for i in range(num_images):
        plt.subplot(1, num_images, i + 1)
        img = x_test[i].numpy().astype("uint8")
        plt.imshow(img)
        plt.title(f"Pred: {class_names[pred_labels[i]]}\nTrue: {class_names[true_labels[i]]}")
        plt.axis('off')
    plt.tight_layout()
    plt.show()

# Show predictions
show_predictions(model, x_test, y_test, class_names, num_images=5)


170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
